In [ ]:
# start coding from here - load grouped data file size 

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt

taxi_grouped_by_region = pd.read_csv('data/taxi_grouped.csv')
taxi_grouped_by_region = taxi_grouped_by_region.drop(labels='Unnamed: 0', axis=1)

print(taxi_grouped_by_region[['transaction_date', 'transaction_month', 'transaction_day', 'transaction_hour']].dtypes)
data_for_benchmark_model = taxi_grouped_by_region.copy()

categorical_features_benchmark = ['PULocationID', 'transaction_month', 'transaction_day', 'transaction_hour']
input_features_benchmark = categorical_features_benchmark
target_features_benchmark = ['total_amount']

X_bench = data_for_benchmark_model[input_features_benchmark]
y_bench = data_for_benchmark_model[target_features_benchmark]
X_bench.dtypes

# one-hot encode
X_bench = pd.get_dummies(X_bench)

# split data
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_bench, y_bench, test_size=0.33, random_state=42)

# fit a model
tree = DecisionTreeRegressor(max_depth=10)
tree.fit(X_train_b, y_train_b)

# model evaluation
model_at_hand = tree
y_pred_b = model_at_hand.predict(X_test_b)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from math import sqrt

print(f'Mean absolute errror: {mean_absolute_error(y_test_b, y_pred_b):.5f}')
print(f'Mean squared errror: {mean_squared_error(y_test_b, y_pred_b):.5f}')
print(f'Root mean squared errror: {sqrt(mean_squared_error(y_test_b, y_pred_b)):.5f}')
print(f'R2 score: {r2_score(y_test_b, y_pred_b):.5f}')

data = {'true': y_test_b.iloc[:, 0], 'predicted': y_pred_b}
# test: DataFrame - predicted: numpy array

results = pd.DataFrame(data)
results.plot(kind='scatter', x='true', y='predicted', figsize=(12, 8))
plt.show()



# 6 Data engineering

In [ ]:
taxi_grouped_by_region.head()

In [ ]:
data_with_new_features = taxi_grouped_by_region.copy()

Data related features

In [ ]:
data_with_new_features['transaction_date'] = pd.to_datetime(data_with_new_features['transaction_date'])
data_with_new_features['transaction_week_day'] = data_with_new_features['transaction_date'].dt.weekday

print(f'Here the week starts from 0 which is Monday')
print(data_with_new_features['transaction_week_day'].value_counts().index)
data_with_new_features['weekend'] = data_with_new_features['transaction_week_day'].apply(lambda x: True if x == 5 | x == 6 else False)

In [ ]:
from datetime import datetime

from pandas.tseries.holiday import USFederalHolidayCalendar
cal = USFederalHolidayCalendar()
holidays = cal.holidays(
    start = datetime(2018, 1, 1), 
    end = datetime(2020, 1, 1)
    )

# transforming numpy array of objects to the DataTime format
holidays = pd.to_datetime(holidays)

data_with_new_features['is_holiday'] = data_with_new_features['transaction_date'].isin(holidays)
data_with_new_features.head()

Borough information

In [ ]:
zone_lookup = pd.read_csv('data/taxi_zone_lookup.csv')
zone_lookup = zone_lookup[['LocationID', 'Borough']]
# zone_lookup['LocationID'] = zone_lookup['LocationID'].astype(str)

print(f"LocationID_type: {zone_lookup['LocationID'].dtype}")
print(f"PULocationID_type: {data_with_new_features['PULocationID'].dtype}")

zone_lookup.loc[264, 'Borough'] = 'Outside of NYC'
zone_lookup.tail()

In [ ]:
data_with_new_features = data_with_new_features.merge(zone_lookup, left_on='PULocationID', right_on='LocationID', how='left')
data_with_new_features = data_with_new_features.drop(labels='LocationID', axis=1)
print(data_with_new_features.shape)
data_with_new_features.head()

In [ ]:
print(data_with_new_features.isna().sum())
data_with_new_features['Borough'].value_counts(dropna=False)

Weather related features

In [ ]:
import requests
import pandas as pd

params = {
    'latitude': 40.7128,
    'longitude': -74.0060,
    'start_date': '2019-01-01',
    'end_date': '2019-01-31',
    'hourly': 'temperature_2m,relative_humidity_2m,cloudcover,windspeed_10m,precipitation',
    'timezone': 'America/New_York'
}

response = requests.get('https://archive-api.open-meteo.com/v1/archive', params=params)

print(response.json().keys())
print(response.json()['hourly'].keys())

if response.status_code == 200:
    data = response.json()
    df = pd.DataFrame(data['hourly'], columns=['time', 'temperature_2m', 'relative_humidity_2m', 'windspeed_10m', 'cloudcover', 'precipitation'])
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    df_3H = df.resample('3h').agg({
        'temperature_2m': 'mean', 
        'relative_humidity_2m': 'mean', 
        'windspeed_10m': 'mean', 
        'cloudcover': 'mean', 
        'precipitation': 'sum'
    })

    df_3H.to_csv('data/nyc_weather_no_missing values.csv')
    print(f'\nWeather data file saved correctly - status code: {response.status_code}')

else:
    print(f'\nSomething went wrong - status code: {response.status_code}')

df_3H.head()

In [ ]:
index = pd.date_range('1/1/2000', periods=9, freq='h')
series = pd.Series(range(9), index=index)
print(series)

series.resample('3h').agg('sum')

In [ ]:
# The source file has been modified to present how to deal with missing values, strings, etc
nyc_weather = pd.read_csv('data/nyc_weather_missing_values.csv')

# if there are NaN values at the beggining or end of DataFrame it would be not possible to use .interpolate()
# use .fillna(method='ffill', 'bfill') instead
nyc_weather.head()
nyc_weather.tail()

In [ ]:
nyc_weather['cloud cover'].value_counts()

In [ ]:
nyc_weather['amount of precipitation'].value_counts()

In [ ]:
nyc_weather.isna().sum()

Trace of precipitation can be thought of as 0.1\
And the missing values 0 ???\
to be considered

In [ ]:
nyc_weather['amount of precipitation'] = nyc_weather['amount of precipitation'].replace('Trace of precipitation', 0.1)
nyc_weather['amount of precipitation'] = nyc_weather['amount of precipitation'] .astype(float)

# Dealing with the missing values

# option 1
# nyc_weather['amount of precipitation'].fillna(0, inplace=True)

# option 2
# nyc_weather['amount of precipitation'].fillna(method='bfill', inplace=True)

# option 3
nyc_weather['amount of precipitation'] = nyc_weather['amount of precipitation'].interpolate()

In [ ]:
mapping = {
    '100%.': 1.0,
    'no clouds': 0.0,
    '70 – 80%.': 0.7,
    '20–30%.': 0.2,
    '50%.': 0.5,
    '??': pd.NA,
    'weird_value1': pd.NA,
    'undefined': pd.NA,
    'broken_data': pd.NA
}

nyc_weather['cloud cover'] = nyc_weather['cloud cover'].replace(mapping)
nyc_weather['cloud cover'] = pd.to_numeric(nyc_weather['cloud cover'], errors='coerce')
nyc_weather['cloud cover'] = nyc_weather['cloud cover'].interpolate()

In [ ]:
nyc_weather.head()
nyc_weather.isna().sum()

In [ ]:
nyc_weather['date and time'] = pd.to_datetime(nyc_weather['date and time'])

nyc_weather['month'] = nyc_weather['date and time'].dt.month
nyc_weather['day'] = nyc_weather['date and time'].dt.day
nyc_weather['hour'] = nyc_weather['date and time'].dt.hour

nyc_weather.head(1)

In [ ]:
data_with_new_features.tail(1)

In [ ]:
data_with_new_features = (data_with_new_features.merge(nyc_weather,
    left_on=['transaction_month','transaction_day','transaction_hour'],
    right_on=['month','day','hour'],
    how='left')
    )
data_with_new_features = data_with_new_features.drop(columns=['date and time', 'month','day','hour'])

print(data_with_new_features.shape)
data_with_new_features.head()

In [ ]:
data_with_new_features.isna().sum()

In [ ]:
data_with_new_features.head()
data_with_new_features.tail(7)

In [ ]:
data_with_new_features.isna().sum()

In [ ]:
# before filling in the missing values it is necessary to sort the data by date and hour
data_with_new_features = data_with_new_features.sort_values(by=['transaction_date', 'transaction_hour']).reset_index(drop=True)

data_with_new_features.head()
data_with_new_features.tail()

In [ ]:
data_with_new_features = data_with_new_features.fillna(method='ffill')
data_with_new_features.tail()

In [ ]:
data_with_new_features.isna().sum()

In [ ]:
def f(x, y):
    print(f'x,y={x}, {y}')
    return x**2 + y**2

a = np.array([2, 3])

val = [f(*a)]

print(f'val={val}')

In [ ]:
# gradient_descent_2d.py
import numpy as np
import matplotlib.pyplot as plt

def f(x, y):
    return x**2 + y**2

def grad_f(x, y):
    return np.array([2*x, 2*y])

def gradient_descent(x0, eta, n_iter):
    
    points = [x0]
    print(f'x0 = {points} type:{x0.dtype}')
    values = [f(*x0)]
    x = np.array(x0, dtype=float)
    for k in range(n_iter):
        x = x - eta * grad_f(*x)
        points.append(x.copy())
        values.append(f(*x))
    return np.array(points), np.array(values)

if __name__ == "__main__":
    # parametry startowe
    x0 = np.array([2.0, 1.5])
    eta = 0.1
    n_iter = 20

    points, values = gradient_descent(x0, eta, n_iter)

    # wypisz pierwsze iteracje
    print("k\t (x_k, y_k)\t\t f(x_k, y_k)")
    for k in range(len(points)):
        print(f"{k}\t ({points[k][0]:.6f}, {points[k][1]:.6f})\t {values[k]:.6f}")

    # rysunek powierzchni + trajektoria
    X, Y = np.meshgrid(np.linspace(-2, 2, 100), np.linspace(-2, 2, 100))
    Z = f(X, Y)

    fig, ax = plt.subplots(figsize=(6,6))
    # poziomice
    cs = ax.contour(X, Y, Z, levels=20, cmap='viridis')
    ax.clabel(cs, inline=True, fontsize=8)

    # trajektoria kroków
    ax.plot(points[:,0], points[:,1], marker='o', color='red')
    ax.set_title("Gradient descent na paraboloidzie")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.axis("equal")
    plt.show()


In [ ]:
# compare_eta.py
import numpy as np
import matplotlib.pyplot as plt

def f(x, y):
    return x**2 + y**2

def grad_f(x, y):
    return np.array([2*x, 2*y])

def gradient_descent_path(x0, eta, n_iter=200, tol=1e-6):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    values = [f(*x)]
    for k in range(n_iter):
        x = x - eta * grad_f(*x)
        path.append(x.copy())
        values.append(f(*x))
        if values[-1] < tol:
            break
    return np.array(path), np.array(values), k+1  # k+1 = liczba wykonanych iteracji

def compare_etas(x0, etas, n_iter=500, tol=1e-8):
    results = []
    all_paths = {}
    all_values = {}
    for eta in etas:
        path, values, iters = gradient_descent_path(x0, eta, n_iter=n_iter, tol=tol)
        final_val = values[-1]
        results.append({'eta': eta, 'iters': iters, 'final_val': final_val, 'last_point': path[-1]})
        all_paths[eta] = path
        all_values[eta] = values
    return results, all_paths, all_values

def print_results_table(results):
    print("eta\titers\tfinal f(x,y)\t\t last point (x,y)")
    for r in results:
        print(f"{r['eta']}\t{r['iters']}\t{r['final_val']:.6e}\t ({r['last_point'][0]:.6f}, {r['last_point'][1]:.6f})")

def plot_contours_with_paths(all_paths, savefile="gd_paths_comparison.png"):
    # siatka poziomic
    limit = 2.5
    X, Y = np.meshgrid(np.linspace(-limit, limit, 300), np.linspace(-limit, limit, 300))
    Z = X**2 + Y**2

    fig, ax = plt.subplots(figsize=(7,7))
    cs = ax.contour(X, Y, Z, levels=30)
    ax.clabel(cs, inline=True, fontsize=8)

    for eta, path in all_paths.items():
        ax.plot(path[:,0], path[:,1], marker='o', label=f"eta={eta}", markersize=4, linewidth=1)
        # punkt startowy (duży)
        ax.scatter(path[0,0], path[0,1], s=60, edgecolors='k', linewidths=0.8)

    ax.set_title("Porównanie trajektorii GD dla różnych eta")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.axis('equal')
    ax.legend()
    plt.tight_layout()
    plt.savefig(savefile, dpi=200)
    plt.close(fig)
    print(f"Zapisano wykres trajektorii do: {savefile}")

def plot_values_over_iters(all_values, savefile="gd_values_comparison.png"):
    fig, ax = plt.subplots(figsize=(8,4))
    for eta, values in all_values.items():
        ax.semilogy(values, marker='o', markersize=4, label=f"eta={eta}")
    ax.set_xlabel("iteracja k")
    ax.set_ylabel("f(x_k,y_k) (log scale)")
    ax.set_title("Spadek wartości funkcji w czasie dla różnych eta")
    ax.legend()
    plt.tight_layout()
    plt.savefig(savefile, dpi=200)
    plt.close(fig)
    print(f"Zapisano wykres wartości funkcji do: {savefile}")

if __name__ == "__main__":
    # punkt startowy
    x0 = np.array([2.0, 1.5])

    # wartości eta do porównania
    etas = [0.05, 0.1, 0.5]

    # parametry eksperymentu
    max_iter = 500
    tol = 1e-8

    results, all_paths, all_values = compare_etas(x0, etas, n_iter=max_iter, tol=tol)

    # wypisz tabelę wyników
    print_results_table(results)

    # zapisz wykresy (kontury + trajektorie; oraz spadek wartości funkcji)
    plot_contours_with_paths(all_paths, savefile="gd_paths_comparison.png")
    plot_values_over_iters(all_values, savefile="gd_values_comparison.png")

    # dodatkowo zapisz dane trajektorii do pliku .npz
    np.savez("gd_compare_data.npz", **{f"path_eta_{eta}": all_paths[eta] for eta in all_paths})
    print("Zapisano dane trajektorii do: gd_compare_data.npz")
